In [17]:
!pip install -r requirements.txt


[notice] A new release of pip is available: 23.0.1 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
import asyncio
from azure.identity.aio import DefaultAzureCredential
from semantic_kernel.agents import AgentGroupChat, AzureAIAgent, AzureAIAgentSettings, ChatCompletionAgent
from dotenv import load_dotenv
import os
from azure.ai.projects.models import AzureAISearchTool, AzureAISearchQueryType, AsyncToolSet
from azure.ai.projects.aio import AIProjectClient

In [35]:
load_dotenv()
project_client = AzureAIAgent.create_client(credential=DefaultAzureCredential(), conn_str=os.getenv("AZURE_AI_CONNECTION_STRING"))
# project_client = AIProjectClient.from_connection_string(
#     credential=DefaultAzureCredential(),
#     conn_str=os.getenv("AZURE_AI_CONNECTION_STRING"),
# )

In [36]:
AGENT_NAME = "RESEARCHER"
INSTRUCTIONS = "You are a helpful assistant that provides information about the surface pro. You do not use your internal knowledge, only use functions available to you to search for data. You can answer questions about the surface pro features and specifications. Provide citation for your answer."

In [37]:
connection = await project_client.connections.get(connection_name=os.environ["AI_SEARCH_CONNECTION_NAME"])
ai_search = AzureAISearchTool(
        index_connection_id=connection.id,
        index_name="surface-pro-index",
        query_type=AzureAISearchQueryType.VECTOR_SEMANTIC_HYBRID,
        top_k=3
)
toolset = AsyncToolSet()
toolset.add(ai_search)

In [38]:
# agent = await project_client.agents.create_agent(
#                 model="gpt-4o",
#                 name=AGENT_NAME,
#                 instructions=INSTRUCTIONS,
#                 toolset=toolset)

azure_ai_agent = AzureAIAgent(
            client=project_client,
            definition=await project_client.agents.create_agent(
                model="gpt-4o",
                name=AGENT_NAME,
                instructions=INSTRUCTIONS,
                toolset=toolset,
            ),
        )

In [39]:
print(f"Created agent, ID: {azure_ai_agent.id}")

Created agent, ID: asst_MtfilsBBrjVM9O7GYZjmMEuE


In [40]:
thread = await project_client.agents.create_thread()

In [44]:
question = "What are the features of the surface pro 9?"
thread = None
async for response in azure_ai_agent.invoke(
                    messages=question,
                    thread=thread,
                ):
                    print(f"# {response.name}: {response}")
                    thread = response.thread

# RESEARCHER: I couldn't find specific information on the Surface Pro 9's features and specifications in the documents provided. You might be interested in checking the official Microsoft website or product guides for the most accurate and detailed information about the Surface Pro 9. If you have any documents or files that might contain the information, feel free to share them, and I can assist in extracting the relevant details.
